# Predictive Detection of Network Intrusions Using Machine Learning
### NSL-KDD Dataset | Network Intrusion Detection System (NIDS)

**Objective:** Build and compare machine-learning classifiers that classify network connections as Normal, DoS, Probe, R2L, or U2R.

**Workflow:** Data understanding → preprocessing → model training → evaluation → interpretation.

> **Dataset setup:** Download the NSL-KDD files from the supplied Kaggle page. Place `KDDTrain+.txt` and `KDDTest+.txt` in the same folder as this notebook. If your downloaded files have different names, update the paths in the next cell.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight

sns.set_theme(style="whitegrid", palette="viridis")
RANDOM_STATE = 42

## 2. Load the NSL-KDD dataset
The standard NSL-KDD files contain 41 input features, an attack label, and a difficulty field. We use the provided train and test files separately to avoid test-data leakage.

In [ ]:
# Change these paths if your downloaded files are stored elsewhere
TRAIN_PATH = "KDDTrain+.txt"
TEST_PATH = "KDDTest+.txt"

columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty"
]

train_df = pd.read_csv(TRAIN_PATH, names=columns)
test_df = pd.read_csv(TEST_PATH, names=columns)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
display(train_df.head())

## 3. Data understanding

In [ ]:
print("Training data information:")
train_df.info()
print("\nMissing values:", train_df.isnull().sum().sum())
print("\nDuplicate rows:", train_df.duplicated().sum())
print("\nNumber of unique attack labels:", train_df["label"].nunique())
print(train_df["label"].value_counts().head(15))

In [ ]:
plt.figure(figsize=(10, 4))
train_df["label"].value_counts().head(15).plot(kind="bar")
plt.title("Most frequent labels in training data")
plt.xlabel("Traffic label")
plt.ylabel("Number of records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Convert attack labels into five categories
The dataset has many individual attack names. This function groups them into the requested five classes. Any label not in the known attack lists is treated as Normal only when it is explicitly named `normal`; unknown labels are reported.

In [ ]:
dos_attacks = {
    "back", "land", "neptune", "pod", "smurf", "teardrop",
    "mailbomb", "apache2", "udpstorm", "processtable", "worm"
}
probe_attacks = {
    "ipsweep", "nmap", "portsweep", "satan", "mscan", " saint"
}
r2l_attacks = {
    "ftp_write", "guess_passwd", "imap", "multihop", "phf",
    "spy", "warezclient", "warezmaster", "sendmail", " named",
    "snmpgetattack", "snmpguess", "xlock", "xsnoop", "worm"
}
u2r_attacks = {
    "buffer_overflow", "loadmodule", "perl", "rootkit",
    "ps", "sqlattack", "xterm"
}

def group_attack(label):
    name = str(label).strip().lower().rstrip(".")
    if name == "normal":
        return "Normal"
    if name in dos_attacks:
        return "DoS"
    if name in probe_attacks:
        return "Probe"
    if name in r2l_attacks:
        return "R2L"
    if name in u2r_attacks:
        return "U2R"
    return "Other"

train_df["target"] = train_df["label"].apply(group_attack)
test_df["target"] = test_df["label"].apply(group_attack)

print("Training grouped labels:")
print(train_df["target"].value_counts())
print("\nUnmapped labels:")
print(sorted(set(train_df.loc[train_df.target == "Other", "label"].unique())))

**Note:** `Other` indicates an attack name not covered by the four requested attack families. Inspect the printed list. Add any valid attack names to the appropriate set above before continuing. The notebook excludes `Other` from model training rather than silently treating it as Normal.

In [ ]:
train_df = train_df[train_df["target"] != "Other"].copy()
test_df = test_df[test_df["target"] != "Other"].copy()

# Remove the original text label and difficulty field; keep the 41 traffic features
X_train = train_df.drop(columns=["label", "difficulty", "target"])
y_train = train_df["target"]
X_test = test_df.drop(columns=["label", "difficulty", "target"])
y_test = test_df["target"]

print("Classes:", sorted(y_train.unique()))
print("Train:", X_train.shape, " Test:", X_test.shape)

## 5. Preprocessing
`protocol_type`, `service`, and `flag` are categorical. One-hot encoding converts them into numeric columns. Numerical features are scaled for SVM. The preprocessing is fitted only on training data.

In [ ]:
categorical_cols = ["protocol_type", "service", "flag"]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

## 6. Handle class imbalance
We inspect the five-class distribution. For the models below, class-balanced sample weights are used during fitting where supported. This helps prevent common classes from dominating training.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=train_df, x="target", order=["Normal", "DoS", "Probe", "R2L", "U2R"])
plt.title("Class distribution after grouping")
plt.xlabel("Class")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

## 7. Train the models
Random Forest and SVM are trained using the same preprocessing pipeline. XGBoost is optional: if it is not installed, install it with `pip install xgboost` and rerun this section.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=150, random_state=RANDOM_STATE,
        class_weight="balanced_subsample", n_jobs=-1
    ),
    "SVM": SVC(
        kernel="rbf", C=1.0, class_weight="balanced"
    )
}

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    xgb_model = XGBClassifier(
        n_estimators=150, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob", eval_metric="mlogloss",
        random_state=RANDOM_STATE, n_jobs=-1
    )
    models["XGBoost"] = xgb_model
    print("XGBoost is available.")
except ImportError:
    print("XGBoost not installed; Random Forest and SVM will run.")

In [ ]:
trained_models = {}
predictions = {}
results = []

for name, model in models.items():
    print("Training:", name)
    if name == "XGBoost":
        Xtr = preprocessor.fit_transform(X_train)
        Xte = preprocessor.transform(X_test)
        model.fit(Xtr, y_train_encoded, sample_weight=sample_weights)
        pred_encoded = model.predict(Xte)
        pred = label_encoder.inverse_transform(pred_encoded.astype(int))
    else:
        pipe = Pipeline([("preprocess", preprocessor), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        model = pipe

    trained_models[name] = model
    predictions[name] = pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (macro)": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score (macro)", ascending=False)
display(results_df.round(4))

## 8. Evaluate each classifier
Macro-averaged metrics give each class equal weight, which is useful when attack classes are imbalanced. The classification report also provides per-class precision, recall, and F1-score.

In [ ]:
for name, pred in predictions.items():
    print("\n", "=" * 60)
    print(name)
    print(classification_report(y_test, pred, labels=["Normal", "DoS", "Probe", "R2L", "U2R"], zero_division=0))

## 9. Confusion matrices

In [ ]:
for name, pred in predictions.items():
    cm = confusion_matrix(y_test, pred, labels=["Normal", "DoS", "Probe", "R2L", "U2R"])
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Normal", "DoS", "Probe", "R2L", "U2R"],
                yticklabels=["Normal", "DoS", "Probe", "R2L", "U2R"], ax=ax)
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    plt.tight_layout()
    plt.show()

## 10. Compare model performance

In [ ]:
metrics = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"]
plot_df = results_df.set_index("Model")[metrics]
ax = plot_df.plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=0)
ax.set_title("Performance comparison of intrusion detection models")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 11. Feature importance (Random Forest)
Feature importance is calculated from the fitted Random Forest model. The preprocessing pipeline expands categorical features, so the transformed feature names are retrieved from the fitted encoder.

In [ ]:
if "Random Forest" in trained_models:
    rf_pipe = trained_models["Random Forest"]
    prep = rf_pipe.named_steps["preprocess"]
    rf = rf_pipe.named_steps["model"]

    feature_names = prep.get_feature_names_out()
    importances = rf.feature_importances_
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x="Importance", y="Feature")
    plt.title("Top 15 Random Forest feature importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
    display(importance_df)

## 12. Predict a sample network connection
This demonstrates how the trained Random Forest pipeline can classify a single record from the test set.

In [ ]:
if "Random Forest" in trained_models:
    sample = X_test.iloc[[0]]
    actual = y_test.iloc[0]
    predicted = trained_models["Random Forest"].predict(sample)[0]
    print("Actual class:", actual)
    print("Predicted class:", predicted)

## 13. Conclusion

This notebook implements a multi-class network intrusion detection workflow using the NSL-KDD dataset. It preprocesses network features, trains Random Forest, SVM, and (when installed) XGBoost, and evaluates them using accuracy, macro precision, macro recall, macro F1-score, classification reports, and confusion matrices.

**Complete this section after running the notebook:** Report the measured scores from the comparison table, describe which attack categories were most often misclassified, and discuss the top features found by Random Forest. Do not claim a model is best until you have examined the actual results.

**Limitations:** NSL-KDD is a benchmark dataset and may not represent current real-world network traffic. High test performance does not by itself establish readiness for deployment. A production NIDS would require validation on contemporary traffic, monitoring for data drift, and careful false-positive analysis.